# W&B Sweep Report

Queries wandb sweep results and builds a summary table + Report. Self-contained: does not depend on `criteo_functions.py` or the other notebooks.

In [ ]:
import wandb
import polars as pl
import pandas as pd
import wandb_workspaces.reports.v2 as wr

In [ ]:
def get_project_sweeps(
    entity: str = "seanv507",
) -> dict[str, list[dict]]:
    """
    Return every accessible project under `entity` and its sweep IDs.

    Projects without sweeps are included with an empty list.
    """
    api = wandb.Api(timeout=60)

    return {
        project.name: [
            {
                "sweep_id": sweep.id,
                "sweep_config": sweep.config,
                "sweep_runs": [{
                    "run_id": run.id,
                    "run_summary_metrics": run.summary_metrics
                } for run in sweep.runs]
            }
            for sweep in project.sweeps(per_page=100)
        ]
        for project in api.projects(entity=entity, per_page=200)
    }

def make_sweeps_df(project_sweeps: dict[str, list[dict]]):
    project_sweeps = (
        pl.DataFrame({'project':project_sweeps.keys(), 'sweeps': project_sweeps.values()},strict=False)
        .explode('sweeps').unnest('sweeps').explode('sweep_runs').unnest('sweep_runs')
        .with_columns(pl.col('run_id'), 
                    (pl.col('sweep_config').struct['parameters']),
                    training_logloss=pl.col('run_summary_metrics').struct['validation_0-logloss.min'],
                    validation_logloss=pl.col('run_summary_metrics').struct['validation_1-logloss.min']
                    )
        .unnest('parameters')
        #.drop_nulls(subset='validation_logloss')
        .sort('validation_logloss')
        
    )
    return project_sweeps

In [ ]:
import re


def get_project_runs(
    entity: str = "seanv507",
    project_name: str | None = None,
) -> dict[str, list[dict]]:
    """
    Return every accessible project under `entity` and its runs, whether or
    not they belong to a sweep.

    `project_name` is an optional regex (matched with `re.search`) used to
    select only projects whose name matches. Projects without runs are
    included with an empty list. Runs outside a sweep have `sweep_id=None`.
    """
    api = wandb.Api(timeout=60)
    pattern = re.compile(project_name) if project_name else None

    return {
        project.name: [
            {
                "run_id": run.id,
                "sweep_id": run.sweep.id if run.sweep else None,
                "state": run.state,
                "name": run.name,
                "created_at": run.created_at,
                "tags": run.tags,
                "notes": run.notes,
                "config": dict(run.config),
                "summary_metrics": {
                    k: v
                    for k, v in run.summary_metrics.items()
                    if isinstance(v, (int, float, str, bool)) or v is None
                },
            }
            for run in api.runs(f"{entity}/{project.name}", per_page=200)
        ]
        for project in api.projects(entity=entity, per_page=200)
        if pattern is None or pattern.search(project.name)
    }


def make_runs_df(project_runs: dict[str, list[dict]]):
    runs = (
        pl.DataFrame(
            {"project": project_runs.keys(), "runs": project_runs.values()},
            strict=False,
        )
        .explode("runs").unnest("runs")
        .drop_nulls(subset="run_id")  # projects with no runs
        .unnest("config", separator=":")
        .unnest("summary_metrics", separator=":")
    )
    return runs


# .with_columns(
        #     training_logloss=pl.col("run_summary_metrics").struct["validation_0-logloss.min"],
        #     validation_logloss=pl.col("run_summary_metrics").struct["validation_1-logloss.min"],
        # )
        # .sort("validation_logloss")

```
run = api.run('seanv507/criteo_boost_sweep_04/auwure38')
```

In [ ]:
ps = get_project_sweeps("seanv507")

In [ ]:
prs = get_project_runs("seanv507", "criteo_boost*")


In [ ]:
project_runs = make_runs_df(prs)

In [ ]:
project_runs.write_excel("project_runs.xlsx")

In [ ]:
print(project_runs.shape)
project_runs.head()

In [ ]:
project_sweeps = make_sweeps_df(ps)

In [ ]:
project_sweeps["sweep_id"].unique()

In [ ]:

project_sweeps.null_count()

In [ ]:
project_sweeps.head()

In [ ]:
project_sweeps.with_columns(pl.col('sweep_config').struct['parameters']).unnest('parameters')

In [ ]:
a1 = project_sweeps.explode('sweeps').unnest('sweeps').explode('sweep_runs').unnest('sweep_runs')


> **⚠️ Syntax error below** — `a2 = a1.` is an incomplete statement (trailing `.` with nothing after it). It looks like it was meant to continue the explode/unnest chain from `a1` above. Complete the expression or remove the cell.

In [ ]:
a2 = a1.
a2.head()

In [ ]:
a2.shape

In [ ]:
a1.shape

In [ ]:
a1.item(0,3)

In [ ]:
project_sweeps_e = project_sweeps.explode("sweep_ids")
project_sweeps_e.head()

In [ ]:
project_sweeps_e.group_by(pl.col("project").str.slice(0,10)).len().sort("len",descending=True)

In [ ]:
ENTITY = "seanv507"
PROJECT = "criteo_boost_sweep"
SWEEP_ID = "s15xvple"
TABLE_KEY = "sweep_column_statistics"

# 1. Read the 45 sweep runs.
api = wandb.Api()
sweep = api.sweep(f"{ENTITY}/{PROJECT}/{SWEEP_ID}")

rows = []
for run in sweep.runs:
    summary = dict(run.summary)
    rows.append({
        "learning_rate": run.config.get("learning_rate"),
        "max_depth": run.config.get("max_depth"),
        "subsample": run.config.get("subsample"),
        # This sweep stores the objective's minimum under this summary key.
        "validation_1-logloss": summary.get(
            "validation_1-logloss.min"
        ),
    })

# 2. Calculate one row per column.
values = pd.DataFrame(rows).apply(pd.to_numeric, errors="coerce")


In [ ]:
values.head()

In [ ]:


stats = (
    values.agg(["min", "max", "mean"])
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)


In [ ]:
stats

In [ ]:

# 3. Log the derived table to a dedicated analysis run.
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    name=f"{SWEEP_ID}-column-statistics",
    job_type="analysis",
) as analysis_run:
    analysis_run.log({
        TABLE_KEY: wandb.Table(dataframe=stats)
    })
    analysis_run_id = analysis_run.id

# 4. Create a Report containing the table.
report = wr.Report(
    entity=ENTITY,
    project=PROJECT,
    title="Criteo boost sweep statistics",
    description=f"Column statistics across all runs in sweep {SWEEP_ID}.",
    blocks=[
        wr.H2("Sweep column statistics"),
        wr.PanelGrid(
            runsets=[
                wr.Runset(
                    entity=ENTITY,
                    project=PROJECT,
                    filters=[
                        wr.Metric("name").isin([analysis_run_id])
                    ],
                )
            ],
            panels=[
                wr.WeavePanelSummaryTable(
                    table_name=TABLE_KEY,
                    layout=wr.Layout(w=24, h=12),
                )
            ],
        ),
    ],
)

report.save()
print(report.url)



> **⚠️ Duplicate** — this cell redoes the entire pipeline from the `ENTITY = ...` cell above through the Report-building cell, self-contained in one cell. Same missing-`pandas`-import issue noted above applies here too.

In [ ]:

ENTITY = "seanv507"
PROJECT = "criteo_boost_sweep"
SWEEP_ID = "s15xvple"
TABLE_KEY = "sweep_column_statistics"

# 1. Read the 45 sweep runs.
api = wandb.Api()
sweep = api.sweep(f"{ENTITY}/{PROJECT}/{SWEEP_ID}")

rows = []
for run in sweep.runs:
    summary = dict(run.summary)
    rows.append({
        "learning_rate": run.config.get("learning_rate"),
        "max_depth": run.config.get("max_depth"),
        "subsample": run.config.get("subsample"),
        # This sweep stores the objective's minimum under this summary key.
        "validation_1-logloss": summary.get(
            "validation_1-logloss.min"
        ),
    })

# 2. Calculate one row per column.
values = pd.DataFrame(rows).apply(pd.to_numeric, errors="coerce")

stats = (
    values.agg(["min", "max", "mean"])
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)

# 3. Log the derived table to a dedicated analysis run.
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    name=f"{SWEEP_ID}-column-statistics",
    job_type="analysis",
) as analysis_run:
    analysis_run.log({
        TABLE_KEY: wandb.Table(dataframe=stats)
    })
    analysis_run_id = analysis_run.id

# 4. Create a Report containing the table.
report = wr.Report(
    entity=ENTITY,
    project=PROJECT,
    title="Criteo boost sweep statistics",
    description=f"Column statistics across all runs in sweep {SWEEP_ID}.",
    blocks=[
        wr.H2("Sweep column statistics"),
        wr.PanelGrid(
            runsets=[
                wr.Runset(
                    entity=ENTITY,
                    project=PROJECT,
                    filters=[
                        wr.Metric("name").isin([analysis_run_id])
                    ],
                )
            ],
            panels=[
                wr.WeavePanelSummaryTable(
                    table_name=TABLE_KEY,
                    layout=wr.Layout(w=24, h=12),
                )
            ],
        ),
    ],
)

report.save()
print(report.url)
